# 19 - Continuous Actions In, Discrete DQN Actions Out

Train a DQN on [Gymnasium Pendulum-v1](https://gymnasium.farama.org/environments/classic_control/pendulum/).
The environment takes a **continuous vector of shape `(1,)`**, containing torque in `[-2, 2]`.
Its observation is `[cos(theta), sin(theta), angular_velocity]`.

One `RegressionHead` produces **33 Q-values**. Each discrete category selects a complete row of a
fixed `action_vectors` table of shape `[33, 1]`. The table includes both limits and zero. The
environment receives `action_vectors[action_id]`; the transformer receives the **actual numeric
torque** in the next history row. The category ID is used only by the objective, never as a
transformer input. DQN chooses a category by epsilon-greedy selection over Q-values; it does not
need a separate policy head or a continuous actor.

```text
history of (previous torque, reward, current observation)
    -> transformer -> 33 Q-values -> one category ID
    -> action_vectors[ID] -> env.step(continuous vector)
    -> append the applied torque, reward, and next observation
```

This is an executable wiring and training example, not a trained swing-up agent or an in-context
adaptation benchmark. It uses a small transformer built from scratch, a local token vocabulary,
and CPU training: no model downloads, Box2D, or MuJoCo are needed. Install the repository's
`examples` extra first. Each cycle collects complete episodes into replay, then trains with Double
DQN and a delayed target model. Replay is retained across cycles.

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from tokenizers import Tokenizer as TextTokenizer
from tokenizers.models import WordLevel
from transformers import PreTrainedTokenizerFast

from mouse_gym import EnvConfig, make_env
from mouse_core import AdamW
from mouse_core.data import DataLoader, Datastore, Tokenizer, save_tokenizer, to_device
from mouse_core.models import Model, model_polyak, save_model
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import RegressionHead
from mouse_core.objectives import DqnObjective, boundary_discount

SEED = 7
NUM_ACTIONS = 33
MAX_EPISODE_STEPS = 200
NUM_CYCLES = 2
EPISODES_PER_CYCLE = 2
TRAIN_STEPS = 2
EPSILON = 0.2
POLYAK_TAU = 0.01
HIDDEN_DIM = 32
NUM_FREQUENCIES = 4
LEARNING_RATE = 3e-4

device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.manual_seed(SEED)
action_rng = np.random.default_rng(SEED)

## Environment and action lookup table

The environment keeps its continuous `Box` action space. The conversion happens in the collector,
so the model's discrete output and the physical action are both visible.
`action_vectors[k]` is a complete action vector, even though Pendulum has just one component.
For a multidimensional environment the same lookup would use a `[K, action_dim]` table, still with
one categorical output over its `K` rows. A Cartesian grid would have the product of the per-axis
bin counts; an explicitly chosen finite list of vectors need not contain every combination.

The table and its ordering are part of the policy. Changing either after training changes what
its outputs mean. A checkpoint must carry this table as well as the network and tokenizer.

In [ ]:
env = make_env(EnvConfig(
    id="Pendulum-v1",
    seed=SEED,
    seed_mode="stream",
    kwargs={"max_episode_steps": MAX_EPISODE_STEPS},
))
assert env.action_space.shape == (1,)
action_vectors = np.linspace(
    env.action_space.low[0], env.action_space.high[0], NUM_ACTIONS, dtype=np.float32,
)[:, None]
action_vectors.setflags(write=False)
assert action_vectors.shape == (NUM_ACTIONS, 1)
assert all(env.action_space.contains(vector) for vector in action_vectors)
assert action_vectors[NUM_ACTIONS // 2, 0] == 0.0

for action_id in (0, NUM_ACTIONS // 2, NUM_ACTIONS - 1):
    print(f"category {action_id:2d} -> continuous vector {action_vectors[action_id]}")

## Numeric history, discrete training target

Each row contains the transition **into** its observation:

| Row field | Transformer input | DQN objective |
| --- | --- | --- |
| `torque` | Applied continuous torque, as a numeric token | No |
| `observation` | Three numeric tokens | No |
| `reward` | Numeric token | Reward |
| `reset_frame` | Numeric flag: 1 on the initial observation, otherwise 0 | No |
| `action_id` | **No** | Copied to objective column `action` |
| `terminated`, `truncated` | No | Boundary discounts |
| `episode_index` | No | Groups the history and loss by episode |

Pendulum's vector has one component, so the collector stores that physical value as scalar
`torque`. It never replaces it with the category ID. A reset frame has dummy torque/ID values;
its flag marks that no action was applied. DQN uses row `t+1`'s ID and reward with row `t`'s
prediction, so the reset row's dummy ID is never a training target.

The small text vocabulary supplies only **field identities**, such as `torque` and `cos`.
Actual floating-point values travel in numeric tensors. Existing Fourier features turn these
values into additions to the field embeddings. The Fourier windows below are wider than the
physical ranges so opposite physical limits do not coincide at the periodic encoding endpoints.

In [ ]:
vocabulary = {
    "[UNK]": 0, "torque": 1, "cos": 2, "sin": 3,
    "velocity": 4, "reward": 5, "reset": 6, "readout": 7,
}
text_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=TextTokenizer(WordLevel(vocab=vocabulary, unk_token="[UNK]")),
    unk_token="[UNK]",
)

tokenizer = Tokenizer(
    tokenizer=text_tokenizer,
    input_fields=[
        {"type": "numeric", "input_field": "torque", "format": "torque",
         "fourier_min": -4.0, "fourier_max": 4.0},
        {"type": "numeric", "input_field": "observation", "input_index": 0,
         "format": "cos", "fourier_min": -2.0, "fourier_max": 2.0},
        {"type": "numeric", "input_field": "observation", "input_index": 1,
         "format": "sin", "fourier_min": -2.0, "fourier_max": 2.0},
        {"type": "numeric", "input_field": "observation", "input_index": 2,
         "format": "velocity", "fourier_min": -16.0, "fourier_max": 16.0},
        {"type": "numeric", "input_field": "reward", "format": "reward",
         "fourier_min": -32.0, "fourier_max": 32.0},
        {"type": "numeric", "input_field": "reset_frame", "format": "reset",
         "fourier_min": -2.0, "fourier_max": 2.0},
        {"type": "token", "format": "readout", "head_output": True},
    ],
    objective_fields=[
        {"input_field": "action_id", "output_field": "action"},
        {"input_field": "reward"},
        {"input_field": "terminated"},
        {"input_field": "truncated"},
    ],
)

## One discrete action-value head

The single head has one Q-value per table row. Its output is a vector of scores over discrete
choices, not a predicted continuous torque. This is ordinary discrete Double DQN; the environment
adapter gives those discrete choices physical meaning.

The reference training kernel supports CPU backward. This short example recomputes the complete
episode prefix for each decision. Larger runs should use the cached rollout pattern from `03`,
closing caches before optimizer updates. The delayed model copies both the backbone and head;
Polyak updates move those copies toward the online model after each optimizer step.

In [ ]:
backbone = TransformerBackbone(
    architecture="qwen3",
    hidden_dim=HIDDEN_DIM,
    num_layers=1,
    num_heads=2,
    vocab_size=len(vocabulary),
    num_frequencies=NUM_FREQUENCIES,
    train_kernel="reference",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
)
head = RegressionHead(
    in_features=HIDDEN_DIM, out_features=NUM_ACTIONS, hidden_dim=HIDDEN_DIM,
    num_layers=1, scale=0.1, use_norm=True, propagate_gradient=1.0,
)
model = Model(
    backbone=backbone,
    heads={"action_value": head},
    action_source="action_value",
    reasoner=None,
).to(device)
delayed_model = model.copy(heads=True, backbone=True, reasoner=False).eval()
update_delayed = model_polyak(online=model, delayed=delayed_model)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters; {NUM_ACTIONS} Q-values")

## Collect complete episodes into replay

Mouse-gym's first step and the step after a boundary are reset frames: their passed-in action is
ignored. We request those frames explicitly, then select actual actions only after observing a
state. Epsilon exploration samples a **category ID**, so all executed actions belong to the table.
Each action is decoded once; the same torque sent to the environment is stored in history.

At the final observation, stop collecting that episode. Keep the final row for value bootstrapping.
The next episode starts a separate attention/loss group. Replay retains earlier cycles' episodes.

In [ ]:
def make_row(*, output, episode_index, action_id, torque, reset_frame):
    observation, reward, terminated, truncated, _info = output
    return {
        "action_id": int(action_id),
        "torque": float(torque),
        "observation": np.asarray(observation, dtype=np.float32).copy(),
        "reward": float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "reset_frame": float(reset_frame),
        "episode_index": int(episode_index),
    }


@torch.no_grad()
def collect_episode(*, episode_index, store):
    model.eval()
    reset_output = env.step(np.zeros(env.action_space.shape, dtype=np.float32))
    assert env.step_index == 0
    history = [make_row(
        output=reset_output, episode_index=episode_index,
        action_id=0, torque=0.0, reset_frame=True,
    )]
    store.append(data=history[0])
    episode_return = 0.0
    while True:
        inputs = tokenizer.pack_rows(rows=[history], continuing=None)
        out = model(inputs)
        q_values = out.predictions["action_value"][-1]
        if action_rng.random() < EPSILON:
            action_id = int(action_rng.integers(NUM_ACTIONS))
        else:
            action_id = int(q_values.argmax().item())
        continuous_action = action_vectors[action_id].copy()
        assert env.action_space.contains(continuous_action)
        output = env.step(continuous_action)
        row = make_row(
            output=output, episode_index=episode_index,
            action_id=action_id, torque=continuous_action[0], reset_frame=False,
        )
        history.append(row)
        store.append(data=row)
        episode_return += row["reward"]
        if row["terminated"] or row["truncated"]:
            return episode_return, history

## Train with the existing Double DQN objective

`action_id` is mapped to the objective's integer `action` column. All 33 categories represent
valid vectors, so no action mask is needed. `double=True` makes online Q choose the next category
and delayed Q evaluate it. No continuous-action maximization or new loss is necessary.

Pendulum ends at a **time limit**, not a terminal physical state. Here `gamma_truncated=1` retains
bootstrapping from the final observation; `gamma_terminated=0` would stop it at a true terminal.
`cross_group_backups="bootstrap"` permits that last-state value. Group boundaries still prevent
attention and transition pairs from crossing into the reset observation of the next episode.

In [ ]:
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=0.99, gamma_terminated=0.0, gamma_truncated=1.0,
    ),
    temperature=0.0,
    double=True,
    cross_group_backups="bootstrap",
    gate=None,
)
optimizer = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)


def train_cycle(*, store, cycle):
    loader = DataLoader(
        stores=[store],
        samples_budget=(MAX_EPISODE_STEPS + 1) * EPISODES_PER_CYCLE,
        batch_size=1,
        sample_field="episode_index",
        transform=tokenizer,
        num_workers=0,
        seed=SEED + cycle,
    )
    try:
        model.train()
        for update in range(TRAIN_STEPS):
            inputs, objective_data, group_id = loader.next_batch()
            out = model(inputs)
            with torch.no_grad():
                delayed_out = delayed_model(inputs)
            losses, metrics = objective(
                objective_data=to_device(data=objective_data, device=device),
                group_id=group_id.to(device),
                predictions=out.predictions["action_value"],
                delayed_predictions=delayed_out.predictions["action_value"],
            )
            loss = torch.stack(tuple(losses.values())).sum()
            assert torch.isfinite(loss)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
            optimizer.step()
            update_delayed(tau_heads=POLYAK_TAU, tau_backbone=POLYAK_TAU)
        return float(loss.detach()), metrics
    finally:
        loader.close()

In [ ]:
store = Datastore(name="pendulum-replay")
training_log = []
try:
    for cycle in range(NUM_CYCLES):
        returns = []
        for episode in range(EPISODES_PER_CYCLE):
            episode_return, last_history = collect_episode(
                episode_index=cycle * EPISODES_PER_CYCLE + episode, store=store,
            )
            returns.append(episode_return)
        loss, metrics = train_cycle(store=store, cycle=cycle)
        training_log.append({"cycle": cycle, "mean_return": float(np.mean(returns)), "loss": loss})
        print(
            f"cycle={cycle} | mean return={np.mean(returns):.1f} | "
            f"loss={loss:.4f} | mean Q={float(metrics['q_values_mean']):.3f} | replay rows={len(store)}"
        )
finally:
    env.close()

print(f"Finished {NUM_CYCLES * EPISODES_PER_CYCLE} episodes and {NUM_CYCLES * TRAIN_STEPS} updates.")

## Inspect the two representations

Changing only `action_id` must not change transformer input tokens or numeric values. Changing
torque must change the numeric input even when the category ID is held fixed. This check also
confirms that the recorded physical action agrees with the lookup table.

The Fourier projection is initialized to zero by mouse-core to preserve pretrained embeddings.
It learns during optimization; the check below verifies that it received a nonzero update.

In [ ]:
example = last_history[1]
assert example["torque"] == float(action_vectors[example["action_id"], 0])
encoded = tokenizer(example)
id_changed = tokenizer({**example, "action_id": (example["action_id"] + 1) % NUM_ACTIONS})
np.testing.assert_array_equal(encoded.ids, id_changed.ids)
np.testing.assert_array_equal(encoded.values, id_changed.values)
assert encoded.objective_fields["action"] != id_changed.objective_fields["action"]

low = tokenizer({**example, "torque": -2.0})
high = tokenizer({**example, "torque": 2.0})
np.testing.assert_array_equal(low.ids, high.ids)
assert low.values[0] == -0.5 and high.values[0] == 0.5
assert model.backbone.fourier.proj.weight.detach().abs().sum().item() > 0.0
print({"category_for_loss": example["action_id"], "torque_for_transformer": example["torque"]})
print("Confirmed: continuous numeric input, discrete objective target.")

## Save the action table with the model (optional)

The existing model serializer saves the network, not environment-specific action decoding.
Set `SAVE_ARTIFACTS=True` to save a local bundle. The action table must be reloaded in the same
row order at inference. The local text tokenizer and the mouse-core packing specification are
saved separately because both use a file named `tokenizer.json`.

Load `model/` with `load_model`, load `text_tokenizer/` with `PreTrainedTokenizerFast.from_pretrained`,
and pass that tokenizer to `load_tokenizer(repo_id_or_path=.../packing)`. Load `action_vectors.npy`
with `np.load`; every sampled ID is decoded by indexing that array. The general inference notebook
`15` would need this decoding step before it could drive Pendulum.

In [ ]:
SAVE_ARTIFACTS = False
OUTPUT_DIR = Path("runs/pendulum-discrete")

if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_model(model=model, path=OUTPUT_DIR / "model")
    text_tokenizer.save_pretrained(OUTPUT_DIR / "text_tokenizer")
    save_tokenizer(tokenizer=tokenizer, path=OUTPUT_DIR / "packing")
    np.save(OUTPUT_DIR / "action_vectors.npy", action_vectors)
    (OUTPUT_DIR / "environment.json").write_text(json.dumps({
        "id": "Pendulum-v1",
        "max_episode_steps": MAX_EPISODE_STEPS,
        "action_vectors": "action_vectors.npy",
        "action_shape": list(action_vectors.shape[1:]),
        "action_dtype": str(action_vectors.dtype),
        "action_columns": ["torque"],
    }, indent=2) + "\n")
    print(f"Saved model, tokenizers, environment settings, and action table to {OUTPUT_DIR.resolve()}")

## What extra machinery did this require?

1. **A fixed action table and a decoder:** the selected integer chooses one complete continuous vector.
2. **Both action representations in collection:** the ID goes to DQN's gather operation; the actual
   continuous components go into the next transformer history row. Keep the table ordering fixed
   during collection, optimization, and inference.
3. **Numeric action input fields:** reuse `Tokenizer(type="numeric")` and the backbone's
   `num_frequencies`, just as for continuous observations. No new embedding API is required.
4. **A checkpoint sidecar:** persist the action table and environment settings alongside the model.

No new loss or shared-package API is needed. Replay, the delayed model, Double DQN, and Polyak
updates are existing components. Arbitrary offline continuous actions would need an encoder
(for example, nearest-vector lookup) and an explicit policy for quantization error; this collector
always selects table rows and needs no inverse mapping. A multidimensional table can become large,
and the Q head itself does not know which vectors are neighbors.